<a href="https://colab.research.google.com/github/The-National-Neighborhood-Data-Archive/usage-metrics/blob/lindsay/NaNDA_StatsScraperv03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📦 Step 1: Import libraries and mount Drive
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from google.colab import drive
import re

# Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 📋 Step 2: Define known NaNDA study IDs
nanda_ids = [
    "38567", "38649", "38974", "39093", "39378", "38559", "38598", "38579",
    "38586", "38597", "38585", "38605", "38569", "38528", "38580", "38584",
    "38606", "38506", "38858"
]

base_url = "https://archive.icpsr.umich.edu/nanda/view/studies/"
results = []

In [ ]:
# 🕸️ Step 3: Scrape study-level data
print("🔍 Scraping individual study data...")
for study_id in nanda_ids:
    url = f"{base_url}{study_id}/study-details"
    print(f"Scraping {url}...")
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')

    # Dataset name from <div class="hero-title"><h1>
    dataset_name = "Unknown"
    hero_title_div = soup.find("div", class_="hero-title")
    if hero_title_div:
        h1 = hero_title_div.find("h1")
        if h1:
            dataset_name = h1.text.strip()
            prefix = "National Neighborhood Data Archive (NaNDA):"
            if dataset_name.lower().startswith(prefix.lower()):
                dataset_name = dataset_name[len(prefix):].strip()

    # Get downloads and citations
    downloads = "NA"
    citations = "NA"

    figures = soup.find_all("div", attrs={"data-testid": "study-stat-tracker"})
    for fig in figures:
        value = fig.find("span", class_="display-2").text.strip()
        label = fig.find("figcaption").text.strip().lower()
        if "download" in label:
            downloads = value
        elif "citation" in label:
            citations = value

    results.append({
        "study_id": study_id,
        "dataset_name": dataset_name,
        "downloads": downloads,
        "citations": citations
    })

    time.sleep(1)

🔍 Scraping individual study data...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38567/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38649/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38974/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/39093/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/39378/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38559/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38598/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38579/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38586/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38597/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view/studies/38585/study-details...
Scraping https://archive.icpsr.umich.edu/nanda/view

In [ ]:
# 🆕 NEW: Step 4: Scrape publications data
print("\n📚 Scraping NaNDA publications data...")
publications = []
publications_base_url = "https://search.icpsr.umich.edu/search/search/nanda/publications"

# Start with the first page
start = 0
rows = 50  # Results per page
has_more_pages = True

while has_more_pages:
    pub_url = f"{publications_base_url}?start={start}&__COMPOUND_SCOPE__=(SERIESID%3A1920%20OR%20ARCHIVE%3Ananda)%20AND%20PUBLISH_STATUS%3APUBLISHED&sort=TITLE_SORT%20asc&rows={rows}"
    print(f"  📄 Scraping publications page {start//rows + 1} (items {start+1}-{start+rows})...")

    try:
        pub_response = requests.get(pub_url)
        pub_soup = BeautifulSoup(pub_response.text, 'html.parser')

        # Find publication entries (we'll need to inspect the page to get the right selectors)
        # First, let's do some diagnostics to see what's actually on the page
        print(f"    🔍 Page title: {pub_soup.title.string if pub_soup.title else 'No title'}")

        # Look at the page structure
        all_divs = pub_soup.find_all("div", class_=True)
        unique_classes = set()
        for div in all_divs[:20]:  # Check first 20 divs
            if div.get('class'):
                unique_classes.update(div.get('class'))

        print(f"    🔍 Found div classes: {list(unique_classes)[:10]}")  # Show first 10 classes

        # The page loads data via JavaScript/React with JSON embedded in the page
        # Look for the JSON data in the page source
        import json

        page_text = pub_response.text

        # Find the JSON data that contains the search results
        json_start = page_text.find('"response":{"docs":[')
        if json_start != -1:
            # Find the end of this JSON structure
            # Look for the searchConfig part that comes after
            json_end = page_text.find('}, searchConfig', json_start)
            if json_end != -1:
                # Extract just the response part
                json_start_full = page_text.rfind('{"response":', 0, json_start + 50)
                json_text = page_text[json_start_full:json_end+1]

                try:
                    data = json.loads(json_text)
                    docs = data.get('response', {}).get('docs', [])
                    total_found = data.get('response', {}).get('numFound', 0)

                    print(f"    🎉 Successfully parsed JSON! Found {len(docs)} docs on this page")
                    print(f"    📊 Total available: {total_found}")

                    for doc in docs:
                        try:
                            title = doc.get('TITLE', 'Unknown')
                            authors = ', '.join(doc.get('AUTHORS_SPLIT', [])) if doc.get('AUTHORS_SPLIT') else 'Unknown'
                            year = doc.get('YEAR_PUB', 'Unknown')
                            publication_type = doc.get('RIS_TYPE', 'Unknown')
                            journal = doc.get('JOURNAL', doc.get('PUBLISHER', 'Unknown'))
                            doi = doc.get('DOI', '')

                            # Get referenced study IDs
                            referenced_studies = doc.get('STUDY_NO', [])
                            if not isinstance(referenced_studies, list):
                                referenced_studies = [referenced_studies]

                            publications.append({
                                "title": title,
                                "authors": authors,
                                "year": year,
                                "publication_type": publication_type,
                                "journal": journal,
                                "doi": doi,
                                "referenced_studies": ", ".join([str(s) for s in referenced_studies]) if referenced_studies else "Unknown",
                                "page_number": start//rows + 1
                            })

                        except Exception as e:
                            print(f"    ⚠️  Error parsing publication doc: {e}")
                            continue

                    # Check if we need more pages
                    current_page_num = start//rows + 1
                    total_pages = (total_found + rows - 1) // rows  # Ceiling division

                    print(f"    📄 Page {current_page_num} of {total_pages}")

                    if current_page_num < total_pages:
                        print(f"    ➡️  Moving to page {current_page_num + 1}")
                        start += rows
                        time.sleep(2)
                    else:
                        print(f"    ✅ Reached final page")
                        has_more_pages = False

                except json.JSONDecodeError as e:
                    print(f"    ❌ Error parsing JSON: {e}")
                    has_more_pages = False
            else:
                print(f"    ❌ Could not find end of JSON data")
                has_more_pages = False
        else:
            print(f"    ❌ Could not find JSON data in page")
            has_more_pages = False

    except Exception as e:
        print(f"    ❌ Error scraping publications page: {e}")
        break

print(f"📊 Found {len(publications)} total publications")



📚 Scraping NaNDA publications data...
  📄 Scraping publications page 1 (items 1-50)...
    🔍 Page title: Search Publications
    🔍 Found div classes: ['offcanvas-header', 'archive-nanda', 'container-fluid', 'w-auto', 'footer-action-bar', 'nav-bg-light', 'footer-copyright', 'footer-logo', 'body-content', 'container']
    🎉 Successfully parsed JSON! Found 50 docs on this page
    📊 Total available: 156
    📄 Page 1 of 4
    ➡️  Moving to page 2
  📄 Scraping publications page 2 (items 51-100)...
    🔍 Page title: Search Publications
    🔍 Found div classes: ['offcanvas-header', 'archive-nanda', 'container-fluid', 'w-auto', 'footer-action-bar', 'nav-bg-light', 'footer-copyright', 'footer-logo', 'body-content', 'container']
    🎉 Successfully parsed JSON! Found 50 docs on this page
    📊 Total available: 156
    📄 Page 2 of 4
    ➡️  Moving to page 3
  📄 Scraping publications page 3 (items 101-150)...
    🔍 Page title: Search Publications
    🔍 Found div classes: ['offcanvas-header', 'arch

In [ ]:
# 💾 Step 5: Save both datasets to your shared drive
output_base_path = "/content/drive/Shared drives/ISR-ISR-NaNDAU01/Metrics/"

if results:
    # Save study-level data
    studies_df = pd.DataFrame(results)
    studies_path = f"{output_base_path}nanda_usage_stats.csv"
    studies_df.to_csv(studies_path, index=False)
    print(f"✅ Study data saved to: {studies_path}")

    # Show preview
    print("\n📊 Study data preview:")
    print(studies_df.head())

if publications:
    # Save publications data
    publications_df = pd.DataFrame(publications)
    publications_path = f"{output_base_path}nanda_publications.csv"
    publications_df.to_csv(publications_path, index=False)
    print(f"✅ Publications data saved to: {publications_path}")

    # Show preview
    print("\n📚 Publications data preview:")
    print(publications_df.head())

    # Show summary stats
    print(f"\n📈 Publications summary:")
    print(f"  • Total publications: {len(publications)}")
    if 'year' in publications_df.columns:
        year_counts = publications_df['year'].value_counts().head()
        print(f"  • Most active years: {dict(year_counts)}")
else:
    print("🚨 No publications data to save.")

    time.sleep(1)

✅ Study data saved to: /content/drive/Shared drives/ISR-ISR-NaNDAU01/Metrics/nanda_usage_stats.csv

📊 Study data preview:
  study_id                                       dataset_name downloads  \
0    38567  Broadband Internet Availability, Speed, and Ad...       125   
1    38649         Crimes by County, United States, 2002-2014       110   
2    38974  Essential Workers by Census Tract and ZIP Code...        31   
3    39093  Home Mortgage Disclosure Act Longitudinal Data...        46   
4    39378  Hospitals by Census Tract and ZIP Code Tabulat...         0   

  citations  
0         3  
1         4  
2         0  
3         3  
4         0  
✅ Publications data saved to: /content/drive/Shared drives/ISR-ISR-NaNDAU01/Metrics/nanda_publications.csv

📚 Publications data preview:
                                               title  \
0  A 5-year longitudinal retrospective cohort stu...   
1  Aging in Places: A Gerontological Investigatio...   
2  Analyzing the relationship between 

In [ ]:
# 💾 Step 4: Save to your shared drive
if results:
    df = pd.DataFrame(results)
    output_path = "/content/drive/Shared drives/ISR-ISR-NaNDAU01/Metrics/nanda_usage_stats.csv"
    df.to_csv(output_path, index=False)
    print(f"✅ File saved to: {output_path}")

    # 🆕 NEW: Display the results to see what we got
    print("\n📊 Results preview:")
    print(df.head())
else:
    print("🚨 No results to save.")

✅ File saved to: /content/drive/Shared drives/ISR-ISR-NaNDAU01/Metrics/nanda_usage_stats.csv

📊 Results preview:
  study_id                                       dataset_name downloads  \
0    38567  Broadband Internet Availability, Speed, and Ad...       125   
1    38649         Crimes by County, United States, 2002-2014       110   
2    38974  Essential Workers by Census Tract and ZIP Code...        31   
3    39093  Home Mortgage Disclosure Act Longitudinal Data...        46   
4    39378  Hospitals by Census Tract and ZIP Code Tabulat...         0   

  citations  
0         3  
1         4  
2         0  
3         3  
4         0  
